In [1]:
!pip install torch torchaudio transformers numpy scipy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.

In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
DATA_ROOT = "/kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
EPOCHS = 20

In [4]:
!pip install pykan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 3.6 MB/s eta 0:00:00


In [36]:
import os
import numpy as np
import torch
import torch.nn as nn
import torchaudio

from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
from sklearn.metrics import roc_curve

In [37]:
DATA_ROOT = "/kaggle/input/asvpoof-2019-dataset"

# Inside this, there is LA/LA/
LA_ROOT = f"{DATA_ROOT}/LA/LA"

TRAIN_DIR = f"{LA_ROOT}/ASVspoof2019_LA_train"
DEV_DIR   = f"{LA_ROOT}/ASVspoof2019_LA_dev"

PROTOCOL_DIR = f"{LA_ROOT}/ASVspoof2019_LA_cm_protocols"

TRAIN_PROTOCOL = f"{PROTOCOL_DIR}/ASVspoof2019.LA.cm.train.trn.txt"
DEV_PROTOCOL   = f"{PROTOCOL_DIR}/ASVspoof2019.LA.cm.dev.trl.txt"

BATCH_SIZE = 4
EPOCHS = 20
SR = 16000

In [38]:
def compute_eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    return eer

In [39]:
def collate_fn(batch):
    audios, labels = zip(*batch)

    audios = torch.nn.utils.rnn.pad_sequence(
        audios, batch_first=True
    )

    # cqccs = torch.stack(cqccs)
    labels = torch.tensor(labels)

    return audios, labels

In [40]:
class ASVspoof2019Dataset(Dataset):
    def __init__(self, base_dir, protocol_file):
        self.base_dir = base_dir
        self.samples = []

        with open(protocol_file) as f:
            for line in f:
                parts = line.strip().split()
                utt_id = parts[1]
                label = 1 if parts[4] == "spoof" else 0
                self.samples.append((utt_id, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        utt_id, label = self.samples[idx]

        audio_path = os.path.join(
            self.base_dir, "flac", utt_id + ".flac"
        )

        waveform, sr = torchaudio.load(audio_path)
        waveform = waveform.squeeze(0)

        return waveform, label

In [41]:
from scipy.fftpack import dct

def extract_cqcc(waveform, n_coeffs=40):
    spectrum = np.abs(np.fft.rfft(waveform))
    log_spec = np.log(spectrum + 1e-6)
    cepstra = dct(log_spec, norm="ortho")
    return torch.tensor(cepstra[:n_coeffs], dtype=torch.float)

In [11]:
def build_cqcc_cache(dataset, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    for i in range(len(dataset)):
        audio, _ = dataset[i]
        cqcc = extract_cqcc(audio.numpy())
        torch.save(cqcc, f"{save_dir}/{i}.pt")

In [12]:
class ASVspoofWithCQCC(Dataset):
    def __init__(self, audio_dataset, cqcc_dir):
        self.audio_dataset = audio_dataset
        self.cqcc_dir = cqcc_dir

    def __len__(self):
        return len(self.audio_dataset)

    def __getitem__(self, idx):
        audio, label = self.audio_dataset[idx]
        cqcc = torch.load(f"{self.cqcc_dir}/{idx}.pt")
        return audio, cqcc, label

In [42]:
class Wav2VecEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-xls-r-300m"
        )
        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.model(x).last_hidden_state

In [43]:
#first trying out linear, keeping the module for later.
class KANProjection(nn.Module):
    def __init__(self, in_dim=1024, out_dim=256):
        super().__init__()
        self.kan = KAN(
            width=[in_dim, out_dim],
            grid=3,   # keep SMALL
            k=3
        )

    def forward(self, x):
        # x: [B, T, 1024]
        B, T, D = x.shape
        x = x.view(B * T, D)
        x = self.kan(x)
        return x.view(B, T, -1)

In [44]:
class LinearProjection(nn.Module):
    def __init__(self, in_dim=1024, out_dim=256):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        # x: [B, T, 1024]
        return self.proj(x)

In [45]:
class TemporalSelfAttention(nn.Module):
    def __init__(self, dim=256, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            batch_first=True
        )

    def forward(self, x):
        # x: [B, T, D]
        out, _ = self.attn(x, x, x)
        return out

In [46]:
class XLSRClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Wav2VecEncoder()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(1024, 1)

    def forward(self, x):
        x = self.encoder(x)          # (B, T, 1024)
        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)
        return torch.sigmoid(self.fc(x))

In [47]:
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        # x: [B, T, D]
        weights = torch.softmax(self.score(x), dim=1)
        return (weights * x).sum(dim=1)

In [48]:
class ClassifierHead(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.fc(x))

In [49]:
class XLSR_Minimal_KANGAT(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-xls-r-300m"
        )
        for p in self.encoder.parameters():
            p.requires_grad = False

        self.proj = LinearProjection(1024,256)
        self.attn = TemporalSelfAttention(dim=256, heads=4)
        self.pool = AttentionPooling(256)
        self.head = ClassifierHead(256)

    def forward(self, audio):
        x = self.encoder(audio).last_hidden_state   # [B, T, 1024]
        x = self.proj(x)                             # [B, T, 256]
        x = self.attn(x)                             # [B, T, 256]
        x = self.pool(x)                             # [B, 256]
        return self.head(x)

In [21]:
class CQCCClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(40, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return torch.sigmoid(self.net(x))

In [22]:
class DualStreamModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.stream_a = XLSRClassifier()
        self.stream_b = CQCCClassifier()

    def forward(self, audio, cqcc):
        p_a = self.stream_a(audio)
        p_b = self.stream_b(cqcc)
        return p_a

In [50]:
print(os.path.exists(TRAIN_PROTOCOL))
print(os.path.exists(DEV_PROTOCOL))
print(len(os.listdir(f"{TRAIN_DIR}/flac")))

True
True
25380


In [51]:
train_audio_ds = ASVspoof2019Dataset(TRAIN_DIR, TRAIN_PROTOCOL)
dev_audio_ds   = ASVspoof2019Dataset(DEV_DIR, DEV_PROTOCOL)

In [25]:
labels = [label for _, label in train_audio_ds]
print("Unique labels:", set(labels))
print("Spoof ratio:", sum(labels) / len(labels))

Unique labels: {0, 1}
Spoof ratio: 0.8983451536643026


In [26]:
import random

indices = random.sample(range(len(train_audio_ds)), 2000)
labels = [train_audio_ds[i][1] for i in indices]

print("Unique labels:", set(labels))
print("Spoof ratio:", sum(labels) / len(labels))

Unique labels: {0, 1}
Spoof ratio: 0.901


In [52]:
with open(TRAIN_PROTOCOL) as f:
    for i in range(5):
        line = f.readline().strip()
        print(line.split())

['LA_0079', 'LA_T_1138215', '-', '-', 'bonafide']
['LA_0079', 'LA_T_1271820', '-', '-', 'bonafide']
['LA_0079', 'LA_T_1272637', '-', '-', 'bonafide']
['LA_0079', 'LA_T_1276960', '-', '-', 'bonafide']
['LA_0079', 'LA_T_1341447', '-', '-', 'bonafide']


In [28]:
build_cqcc_cache(train_audio_ds, "/kaggle/working/cqcc_train")
build_cqcc_cache(dev_audio_ds, "/kaggle/working/cqcc_dev")

In [29]:
len(os.listdir("/kaggle/working/cqcc_train"))

25380

In [30]:
print("Train CQCC files:", len(os.listdir("/kaggle/working/cqcc_train")))
print("Dev CQCC files:", len(os.listdir("/kaggle/working/cqcc_dev")))
print("Train samples:", len(train_audio_ds))
print("Dev samples:", len(dev_audio_ds))

Train CQCC files: 25380
Dev CQCC files: 24844
Train samples: 25380
Dev samples: 24844


In [53]:
from torch.utils.data import DataLoader 

train_loader = DataLoader(
    train_audio_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_audio_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [32]:
train_ds = ASVspoofWithCQCC(train_audio_ds, "/kaggle/working/cqcc_train")
dev_ds   = ASVspoofWithCQCC(dev_audio_ds, "/kaggle/working/cqcc_dev")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn
)

In [54]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device", device)

model = XLSR_Minimal_KANGAT().to(device)

criterion = nn.BCELoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)

using device cuda


In [55]:
EPOCHS = 3

In [56]:
from tqdm.auto import tqdm 

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=True
    )

    for step, (audio, label) in enumerate(pbar): # removed cqcc in between audio and label
        audio = audio.to(device)
        # cqcc = cqcc.to(device)
        label = label.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        out = model(audio)
        loss = criterion(out, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        avg_loss = total_loss / (step + 1)

        # Update tqdm bar
        pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{avg_loss:.4f}"
        })

    print(
        f"Epoch {epoch+1}/{EPOCHS} finished | "
        f"Avg Loss: {total_loss/len(train_loader):.4f}"
    )

Epoch 1/3:   0%|          | 0/6345 [00:00<?, ?it/s]

Epoch 1/3 finished | Avg Loss: 0.3050


Epoch 2/3:   0%|          | 0/6345 [00:00<?, ?it/s]

Epoch 2/3 finished | Avg Loss: 0.2626


Epoch 3/3:   0%|          | 0/6345 [00:00<?, ?it/s]

Epoch 3/3 finished | Avg Loss: 0.2484


In [57]:
model.eval()
scores, labels = [], []

with torch.no_grad():
    for audio, label in dev_loader: # again, removed cqcc inbetween audio and label
        audio = audio.to(device)
        # cqcc = cqcc.to(device)
        out = model(audio)

        scores.extend(out.cpu().numpy())
        labels.extend(label.numpy())

eer = compute_eer(labels, scores)
print("DEV EER:", eer)

KeyboardInterrupt: 

In [58]:
model.eval()
scores, labels = [], []

with torch.no_grad():
    for audio, label in dev_loader:
        audio = audio.to(device)

        out = model(audio)          # shape: [B, 1]
        out = out.squeeze(1)        # shape: [B]

        scores.extend(out.cpu().numpy())
        labels.extend(label.numpy())

eer = compute_eer(labels, scores)
print("DEV EER:", eer)

DEV EER: 0.130298273155416


In [59]:
torch.save(model.state_dict(), "baseline-with-simplifiedKAN.pt")

In [60]:
import zipfile

with zipfile.ZipFile("/kaggle/working/model_bundle.zip", "w") as z:
    z.write("/kaggle/working/baseline-with-simplifiedKAN.pt")

print("Zipped")

Zipped


In [61]:
from IPython.display import FileLink

# specific the file name exactly as it appears
FileLink(r'model_bundle.zip')

/kaggle/working/model_bundle.zip

In [62]:
import os

# 1. Setup your credentials (replace with your actual values)
os.environ['KAGGLE_USERNAME'] = "harshavardhankd"
os.environ['KAGGLE_KEY'] = "KGAT_4f1dfe7313b672af181402cebb8b11ac"

In [63]:
!mkdir -p /kaggle/working/upload_staging
!cp /kaggle/working/model_bundle.zip /kaggle/working/upload_staging/

In [64]:
!kaggle datasets init -p /kaggle/working/upload_staging

Data package template written to: /kaggle/working/upload_staging/dataset-metadata.json


In [65]:
import json
with open('/kaggle/working/upload_staging/dataset-metadata.json', 'r') as f:
    data = json.load(f)

data['title'] = "My Saved Model Bundle"
data['id'] = f"{os.environ['KAGGLE_USERNAME']}/my-saved-model-bundle"

with open('/kaggle/working/upload_staging/dataset-metadata.json', 'w') as f:
    json.dump(data, f)

In [69]:
!kaggle datasets create -p /kaggle/working/upload_staging

Starting upload for file model_bundle.zip
401 Client Error: Unauthorized for url: https://www.kaggle.com/api/v1/blobs/upload


In [68]:
!curl --upload-file model_bundle.zip https://transfer.sh/model_bundle.zip

curl: (7) Failed to connect to transfer.sh port 443 after 149 ms: Connection refused
